# Generative AI Applications — Week 3 Workshop
## Patchify / Unpatchify + Timestep Embedding (ViT/ DiT basics)

**What you’ll practice (hands-on):**
1. Converting images `(B, C, H, W)` into patch tokens `(B, N, patch_dim)` and back.
2. Implementing sinusoidal timestep embeddings and injecting them into token embeddings.
3. **Tensor dimension play:** `reshape`, `view`, `permute`, `contiguous`, broadcasting with `unsqueeze`.


**Focus:** being precise about tensor dimensions - the key to write efficient code and scale up!

---

### To achieve:
- `patchify(x, patch_size)` and `unpatchify(tokens, patch_size, H, W, C)` that are inverses.
- `timestep_embedding(t, dim)` (sin/cos) + `TimeEmbedding` module (MLP projection).
- A short demo showing shapes at each stage and successful reconstructions.

> Tip: When shapes don’t match, print them early and often.

 - This worksheet is prepared with the help of chatGPT.


### Broadcasting

Numpy tutorial: https://numpy.org/doc/2.3/user/basics.broadcasting.html

Pytorch: https://docs.pytorch.org/docs/stable/notes/broadcasting.html

"newaxis" in numpy: https://numpy.org/doc/2.2/reference/constants.html#numpy.newaxis. 

"`numpy.newaxis`: A convenient alias for None, useful for indexing arrays."

In [ ]:
# Setup
import math
import torch
import torch.nn as nn

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


In [ ]:
def shape(x):
    return tuple(x.shape)

def assert_close(a, b, tol=1e-6):
    max_err = (a-b).abs().max().item()
    print("max_err:", max_err)
    assert max_err < tol, f"Max error {max_err} >= {tol}"

def pretty_shapes(**tensors):
    for k, v in tensors.items():
        print(f"{k:>14}: {tuple(v.shape)}")


## Task 1 — Patchify (image → patch tokens)

Implement `patchify`.

**Input:** `x` with shape `(B, C, H, W)`

**Output:** `tokens` with shape `(B, N, patch_dim)` where:
- `patch_dim = C * P * P`
- `N = (H/P) * (W/P)`

Constraints:
- Assume `H` and `W` are divisible by `P`.
- Do **not** use `unfold`; use `reshape/view` + `permute`.


In [ ]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """
    x: (B, C, H, W)
    returns: (B, N, C*P*P)
    """
    # TODO:
    # 1) Split H and W into (H//P, P) and (W//P, P)
    # 2) Reorder dims so patches are grouped as tokens
    # 3) Flatten each patch into length C*P*P
    raise NotImplementedError

# Quick shape test
B, C, H, W = 2, 3, 32, 32
P = 8
x = torch.randn(B, C, H, W, device=device)
tokens = patchify(x, P)
pretty_shapes(x=x, tokens=tokens)

# Expected: tokens is (B, (H/P)*(W/P), C*P*P) = (2, 16, 192)
assert shape(tokens) == (B, (H//P)*(W//P), C*P*P)
print("✅ Task 1 shape check passed")


## Task 2 — Unpatchify (patch tokens → image)

Implement `unpatchify` as the inverse of `patchify`.

**Input:** `tokens` with shape `(B, N, C*P*P)`
**Output:** `x_recon` with shape `(B, C, H, W)`

Hints:
- Recover the 2D patch grid: `h = H//P`, `w = W//P` so `N = h*w`.
- Reshape back into `(B, h, w, C, P, P)` then permute to `(B, C, H, W)`.


In [ ]:
def unpatchify(tokens: torch.Tensor, patch_size: int, H: int, W: int, C: int) -> torch.Tensor:
    """
    tokens: (B, N, C*P*P)
    returns: (B, C, H, W)
    """
    # TODO:
    raise NotImplementedError

# Reconstruction test (should be exact for pure reshape/permute)
x_recon = unpatchify(tokens, P, H, W, C)
pretty_shapes(x=x, x_recon=x_recon)
assert shape(x_recon) == shape(x)
assert_close(x_recon, x)
print("✅ Task 2 reconstruction check passed")


## Task 3 — Sinusoidal timestep embedding

Implement sinusoidal embeddings (like Transformers) for diffusion timesteps.

**Input:** `t` shape `(B,)` (ints or floats), and embedding dimension `dim` (even)  
**Output:** `emb` shape `(B, dim)`

Use:
- first half = `sin(t * freq)`
- second half = `cos(t * freq)`

Make sure broadcasting works.

For `None` in dimensions, please refer to the "Broadcasting" resources in the second cell.

Frequencies:

$
f_i \;=\; \exp\!\left(-\,\ln(\text{max\_period})\,\frac{i}{\text{half}}\right)
\;=\; \text{max\_period}^{-\frac{i}{\text{half}}},
\qquad i=0,1,\ldots,\text{half}-1.
$


In [1]:
def timestep_embedding(t: torch.Tensor, dim: int, max_period: int = 10_000) -> torch.Tensor:
    """
    t: (B,) timesteps
    returns: (B, dim)
    """
    # TODO:
    # 1) Create frequencies with shape (dim//2,) - refer to the equation above
    # 2) Compute args = t[:, None] * freqs[None, :] - broadcasting 
    # 3) Concatenate sin and cos along last dim
    raise NotImplementedError

B = 4
t = torch.tensor([0, 1, 10, 999], device=device)
emb = timestep_embedding(t, dim=64)
pretty_shapes(t=t, emb=emb)
assert shape(emb) == (B, 64)
print("✅ Task 3 shape check passed")


NameError: name 'torch' is not defined

## Task 4 — TimeEmbedding module + broadcasting into tokens

Build a small module that:
1) creates sinusoidal `timestep_embedding(t, time_dim)`
2) projects it with an MLP into `model_dim`
3) adds it to patch token embeddings `(B, N, model_dim)` using broadcasting.

**Key dimension trick:** `t_proj[:, None, :]` to make it `(B, 1, D)` so it broadcasts across `N`.


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, time_dim: int, model_dim: int):
        super().__init__()
        # TODO: 2-layer MLP (time_dim -> 4*time_dim -> model_dim) with SiLU
        raise NotImplementedError

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # returns (B, model_dim)
        # TODO:
        raise NotImplementedError

# Demo: patchify -> linear proj -> add time embedding
B, C, H, W = 2, 3, 32, 32
P = 8
D = 128

x = torch.randn(B, C, H, W, device=device)
t = torch.randint(low=0, high=1000, size=(B,), device=device)

tokens = patchify(x, P)                      # (B, N, patch_dim)
patch_dim = tokens.shape[-1]

proj = nn.Linear(patch_dim, D).to(device)
tok_emb = proj(tokens)                       # (B, N, D)

time_emb = TimeEmbedding(time_dim=64, model_dim=D).to(device)
t_emb = time_emb(t)                          # (B, D)

# TODO: add timestep conditioning to every token via broadcasting
# conditioned = tok_emb + ???
raise NotImplementedError

pretty_shapes(tokens=tokens, tok_emb=tok_emb, t_emb=t_emb, conditioned=conditioned)
assert shape(conditioned) == shape(tok_emb)
print("✅ Task 4 conditioning shape check passed")


## Check

If you have time, verify your broadcasting intuition:

- What is the shape of `t_emb`?
- What is the shape of `t_emb[:, None, :]`?
- Why does adding `(B, 1, D)` to `(B, N, D)` work?


In [ ]:
print("t_emb:", t_emb.shape)
print("t_emb[:, None, :]:", t_emb[:, None, :].shape)

_ = tok_emb + t_emb[:, None, :]
print("✅ Broadcasting works")
